In [ ]:
"""
Author: Sophie A. Liu
Date: 05/28/2026
Purpose: NSF to isolate genes. based on Townes(2022) but updated jupyter-- theirs is outdated
"""

In [ ]:
import squidpy as sq                      # vers. 1.11.0 of scanpy

import torch                              # tensorflow is incompatible with python 3.14
import gpytorch
import pandas as pd
import numpy as np

import matplotlib.pyplot as pl

In [33]:
# working directory
import os
os.chdir("i:/Hu Lab/Sophie/1. Cell death/PD1-9_realign/outs/binned_outputs")

In [ ]:
# loading in data
pd1_vis = sq.read.visium("square_008um", load_images=True)

X = pd1_vis.obsm["spatial"]               # gene expression data
Y = pd1_vis.X                             # spatial coordinates

sz = Y.sum(axis=1)                        # total RNA counts at each coordinate

In [ ]:
# required to convert for ML storing and processing
Y = torch.tensor(Y)
X = torch.tensor(X)
sz = torch.tensor(sz)

In [ ]:
# have to learn the ds for reference. part of their dataloading block
idx = np.arange(pd1_vis.n_obs)

seed = 42                                  # the answer to life, the universe, and everything
np.random.shuffle(idx)                     # no, I will never tire of putting this in my code

train_idx = idx[:int(0.8 * len(idx))]
val_idx   = idx[int(0.8 * len(idx)):]

D = {
    "X": X[train_idx],
    "Y": Y[train_idx],
    "sz": sz[train_idx]
}

Dval = {
    "X": X[val_idx],
    "Y": Y[val_idx],
    "sz": sz[val_idx]
}

In [ ]:
# inducing points ie. summarizing the dataset/reducing complexity for gaussian processes. 
from sklearn.cluster import KMeans
Z = KMeans(D["X"])  

Z = torch.tensor(Z)

In [ ]:
# NSF. replaces SpatialFactorization tool. gaussian process defined
class GPFactor(gpytorch.models.ApproximateGP):                       # cannot avoid approximation for NSF
    def __init__(self, Z):                                           # passing inducing pts
        variational_distribution = gpytorch.variational.CholeskyVariationalDistribution(Z.size(0))
        variational_strategy = gpytorch.variational.VariationalStrategy(
            self,                                                    # cholesky optimizes latent factors using lower triangular matrix
            Z,                                                       # numerically more stable since forces all +, QC of inducing pts
            variational_distribution,
            learn_inducing_locations=True
        )
        super().__init__(variational_strategy)

        self.mean_module = gpytorch.means.ZeroMean()
        self.covar_module = gpytorch.kernels.MaternKernel(nu=1.5)   # covariance function for GP, nearby are similiar behavior

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(           # gives distribution of factors over spatial areas
            self.mean_module(x),
            self.covar_module(x)
        )

In [ ]:
# based on inspection, our data follows a negative binomial distribution wrt.and count dying
# using torch's artificial neural networks
class NSF(torch.nn.Module):
    def __init__(self, X, Y, Z, L=4, theta=1.0):
        super().__init__()

        self.X = X
        self.Y = Y
        self.L = L
        self.theta = theta

        # obtaining num. factors L
        self.factors = torch.nn.ModuleList([GPFactor(Z) for _ in range(L)])

        # gene weights times cross product of num. factors by num. genes W (L × G)
        self.W = torch.nn.Parameter(torch.randn(L, Y.shape[1]) * 0.01)   

    def forward(self, x):
        f_list = []

        for gp in self.factors:
            f_list.append(gp(x).rsample())                          # sampling spatial
        F = torch.stack(f_list, dim=1)  # (N × L)                   # combining the samples

        log_mu = F @ self.W + torch.log(sz[:, None] + 1e-8)         # accounting for seq. depth (z-axis effects)
                                                                    # forcing scaling for pos. counts & linear structure
        return log_mu

In [ ]:
# how likely observed counts of genes are under model prediction
def nb_loglik(y, mu, theta=1.0):                                    # set theta to be noise                  
    eps = 1e-8                                                      # numerical stability so no log(0)
    mu = mu.exp()

    t1 = torch.lgamma(y + theta)                                    # gamma function from NB PMF
    t2 = torch.lgamma(theta)
    t3 = torch.lgamma(y + 1)

    p = theta / (theta + mu + eps)                                  # looking at dispersion

    ll = t1 - t2 - t3
    ll += theta * torch.log(p + eps)                                # dispersion penalty
    ll += y * torch.log(1 - p + eps)                                # fitting obs.

    return ll.sum()